In [29]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models , Input
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K
from tensorflow.keras.optimizers import Adam

In [30]:
def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def r2_metric(y_true, y_pred):
    ss_res = K.sum(K.square(y_true - y_pred))
    ss_tot = K.sum(K.square(y_true - K.mean(y_true)))
    return 1 - ss_res / (ss_tot + K.epsilon())

In [31]:
EPOCHS = 70
BATCH_SIZE = 8
TEST_SIZE = 0.2

In [32]:
order = []
Expirements_Functions=[]

In [33]:
# Data Preparation function :
def preparation(forType , input_name , output_name): # fortype hiya data l wach man model (basic wela multitask)
  X = np.load(input_name)
  y = np.load(output_name)
  y = y.reshape(y.shape[0], y.shape[1])

  # Normalization
  num_samples = X.shape[0]
  X_reshaped = X.reshape(num_samples, -1)
  x_scaler = MinMaxScaler()
  X_scaled = x_scaler.fit_transform(X_reshaped)
  X_scaled = X_scaled.reshape(X.shape)

  y_scaler = MinMaxScaler()
  y_scaled = y_scaler.fit_transform(y)

  X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.05, random_state=42)

  input_shape = X_train.shape[1:]
  output_dim = y_train.shape[1]

  if forType == "multitask":
    y_train = [y_train[:, i] for i in range(6)]
    y_test = [y_test[:, i] for i in range(6)]

  return X_train, X_test, y_train, y_test

In [34]:
# Basic PH CNN
def Ph_basic_cnn(input_shape, output_dim):
    model = models.Sequential()
    model.add(layers.Conv2D(16, (3, 3), activation='relu', padding='same', input_shape=input_shape))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.3))
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Flatten())
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(0.2))
    model.add(layers.Dense(output_dim, activation='linear'))
    return model

# Multi Task PH CNN
def build_ph_multi_task_cnn(input_shape=(5, 5, 36)):
    input_layer = Input(shape=input_shape)

    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(input_layer)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)  # better for small inputs than Flatten
    x = layers.Dropout(0.25)(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)

    outputs = []
    for _ in range(6):
        branch = layers.Dense(32, activation='relu')(x)
        branch = layers.Dense(1, activation='linear')(branch)
        outputs.append(branch)

    model = Model(inputs=input_layer, outputs=outputs)
    model.compile(optimizer='adam', loss='mse', metrics=[
         ['mae', rmse, r2_metric],
         ['mae', rmse, r2_metric],
         ['mae', rmse, r2_metric],
         ['mae', rmse, r2_metric],
         ['mae', rmse, r2_metric],
         ['mae', rmse, r2_metric],
    ])
    return model

In [35]:
# Basic A% CNN
def A_basic_cnn(input_shape, output_dim):
    model = models.Sequential()
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.Dense(output_dim, activation='linear'))
    return model

# Multi Task A% CNN
def build_A_multi_task_cnn(input_shape=(5, 5, 36)):
    input_layer = Input(shape=input_shape)

    x = layers.Conv2D(32, (3, 3), padding='same', activation='softmax')(input_layer)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(64, (3, 3), padding='same', activation='linear')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)  # better for small inputs than Flatten
    x = layers.Dropout(0.25)(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)

    outputs = []
    for _ in range(6):
        branch = layers.Dense(64, activation='softmax')(x)
        branch = layers.Dropout(0.25)(branch)
        branch = layers.Dense(128, activation='linear')(x)
        branch = layers.Dropout(0.4)(branch)
        branch = layers.Dense(1, activation='linear')(branch)
        outputs.append(branch)

    model = Model(inputs=input_layer, outputs=outputs)
    model.compile(optimizer='adam', loss='mse', metrics=[
        ['mae', rmse, r2_metric],
        ['mae', rmse, r2_metric],
        ['mae', rmse, r2_metric],
        ['mae', rmse, r2_metric],
        ['mae', rmse, r2_metric],
        ['mae', rmse, r2_metric],
    ])
    return model

In [ ]:
# Basic CE CNN
def CE_basic_cnn(input_shape, output_dim):
    model = models.Sequential()
    
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))

    model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.3))

    # Dense Layers
    model.add(layers.Flatten())

    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(0.3))

    # Output Layer
    model.add(layers.Dense(output_dim, activation='linear'))

    return model

# Multi Task CE CNN
def build_ce_multi_task_cnn(input_shape=(5, 5, 36)):
    input_layer = Input(shape=input_shape)

    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(input_layer)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)

    outputs = []
    for _ in range(6):
        branch = layers.Dense(32, activation='relu')(x)
        branch = layers.Dense(1, activation='linear')(branch)
        outputs.append(branch)

    model = Model(inputs=input_layer, outputs=outputs)
    optimizer = Adam(learning_rate=1e-4)
    model.compile(optimizer=optimizer, loss='mse', metrics=[
        ['mae', rmse, r2_metric],
        ['mae', rmse, r2_metric],
        ['mae', rmse, r2_metric],
        ['mae', rmse, r2_metric],
        ['mae', rmse, r2_metric],
        ['mae', rmse, r2_metric],
    ])
    return model

In [37]:
def fit_models(input_file , output_file , component_name , method , epochs , batch_size , validation_size):
  position = order.index(component_name)
  X_train, X_test, y_train, y_test = preparation(method , input_file , output_file)

  if method =="basic":
    input_shape = X_train.shape[1:]
    output_dim = y_train.shape[1]
    model = Expirements_Functions[position][0](input_shape, output_dim)
    model.compile(optimizer='adam', loss="mse", metrics=['mae', rmse, r2_metric])

  elif method =="multitask":
    model = Expirements_Functions[position][1]()

  history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=validation_size)
  return history

In [38]:
import os
import csv
import re

def csv_saving(component_name , aug , norm , window_size , history):
  final = {k: v[-1] for k, v in history.history.items()}

  layer_names = sorted(set(
      re.match(r"(?:val_)?(dense_\d+)_mae", k).group(1)
      for k in final.keys()
      if re.match(r"(?:val_)?dense_\d+_mae", k)
  ))

  csv_file = "model_results3.csv"
  write_header = not os.path.exists(csv_file)
  with open(csv_file, mode="a", newline="") as f:
      fieldnames = [
          "component", "task", "norm", "aug","mae", "mse", "rmse", "r2",
          "val_mae", "val_mse", "val_rmse", "val_r2",
          "window_size"
      ]
      writer = csv.DictWriter(f, fieldnames=fieldnames)
      if write_header:
          writer.writeheader()
      for layer in layer_names:
          row = {
              "component": component_name,
              "task": layer,
              "norm": norm,
              "aug": aug,
              "mae": final.get(f"{layer}_mae", ""),
              "mse": final.get(f"{layer}_loss", ""),
              "rmse": final.get(f"{layer}_rmse", ""),
              "r2": final.get(f"{layer}_r2_metric", ""),
              "val_mae": final.get(f"val_{layer}_mae", ""),
              "val_mse": final.get(f"val_{layer}_loss", ""),
              "val_rmse": final.get(f"val_{layer}_rmse", ""),
              "val_r2": final.get(f"val_{layer}_r2_metric", ""),
              "window_size": window_size
          }
          writer.writerow(row)

In [39]:
order.append("PH")
Expirements_Functions.append((Ph_basic_cnn , build_ph_multi_task_cnn))
order.append("A%")
Expirements_Functions.append((A_basic_cnn , build_A_multi_task_cnn))
order.append("CE")
Expirements_Functions.append((CE_basic_cnn , build_ce_multi_task_cnn))

In [40]:
%cd /content/drive/MyDrive/CNN DATA

/content/drive/MyDrive/CNN DATA


In [41]:
history = fit_models("aug_5x5.npy" , "aug/outputpH.npy" , "PH" , "multitask" , EPOCHS , BATCH_SIZE , TEST_SIZE)

Epoch 1/70
303/303 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - dense_105_loss: 0.1302 - dense_105_mae: 0.2699 - dense_105_r2_metric: -133.4910 - dense_105_rmse: 0.3271 - dense_107_loss: 0.1376 - dense_107_mae: 0.2758 - dense_107_r2_metric: -130.5391 - dense_107_rmse: 0.3337 - dense_109_loss: 0.1405 - dense_109_mae: 0.2751 - dense_109_r2_metric: -230.6328 - dense_109_rmse: 0.3435 - dense_111_loss: 0.1405 - dense_111_mae: 0.2865 - dense_111_r2_metric: -136.8583 - dense_111_rmse: 0.3683 - dense_113_loss: 0.1858 - dense_113_mae: 0.3189 - dense_113_r2_metric: -147.4965 - dense_113_rmse: 0.4164 - dense_115_loss: 0.1728 - dense_115_mae: 0.3125 - dense_115_r2_metric: -183.2891 - dense_115_rmse: 0.4117 - loss: 0.9073 - val_dense_105_loss: 0.0253 - val_dense_105_mae: 0.1270 - val_dense_105_r2_metric: -19.3990 - val_dense_105_rmse: 0.1688 - val_dense_107_loss: 0.0210 - val_dense_107_mae: 0.1103 - val_dense_107_r2_metric: -13.9688 - val_dense_107_rmse: 0.1539 - val_dense_109_loss: 0.0280 - val_dense_109_

In [42]:
csv_saving("Ph" , "Yes" , "Full" , "5x5" , history)

In [43]:
history = fit_models("aug_5x5.npy" , "aug/outputOM%.npy" , "PH" , "multitask" , EPOCHS , BATCH_SIZE , TEST_SIZE)

Epoch 1/70
303/303 ━━━━━━━━━━━━━━━━━━━━ 16s 19ms/step - dense_118_loss: 0.0611 - dense_118_mae: 0.1575 - dense_118_r2_metric: -223.2783 - dense_118_rmse: 0.1968 - dense_120_loss: 0.0589 - dense_120_mae: 0.1570 - dense_120_r2_metric: -281.7077 - dense_120_rmse: 0.2057 - dense_122_loss: 0.0620 - dense_122_mae: 0.1693 - dense_122_r2_metric: -437.9887 - dense_122_rmse: 0.2177 - dense_124_loss: 0.0465 - dense_124_mae: 0.1404 - dense_124_r2_metric: -429.2046 - dense_124_rmse: 0.1848 - dense_126_loss: 0.0909 - dense_126_mae: 0.1874 - dense_126_r2_metric: -6845.6792 - dense_126_rmse: 0.2335 - dense_128_loss: 0.0263 - dense_128_mae: 0.1074 - dense_128_r2_metric: -2046.7697 - dense_128_rmse: 0.1371 - loss: 0.3457 - val_dense_118_loss: 0.0079 - val_dense_118_mae: 0.0504 - val_dense_118_r2_metric: -11.7362 - val_dense_118_rmse: 0.0656 - val_dense_120_loss: 0.0088 - val_dense_120_mae: 0.0468 - val_dense_120_r2_metric: -13.3288 - val_dense_120_rmse: 0.0726 - val_dense_122_loss: 0.0123 - val_dense_12

In [44]:
csv_saving("OM" , "Yes" , "Full" , "5x5" , history)

In [45]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [46]:
history = fit_models("aug_5x5.npy" , "aug/outputA%.npy" , "A%" , "multitask" , EPOCHS , BATCH_SIZE , TEST_SIZE)

Epoch 1/70
303/303 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - dense_132_loss: 0.0601 - dense_132_mae: 0.1577 - dense_132_r2_metric: -6206.0430 - dense_132_rmse: 0.2000 - dense_135_loss: 0.0724 - dense_135_mae: 0.1680 - dense_135_r2_metric: -5147.5483 - dense_135_rmse: 0.2101 - dense_138_loss: 0.0976 - dense_138_mae: 0.1955 - dense_138_r2_metric: -3942.9326 - dense_138_rmse: 0.2420 - dense_141_loss: 0.0949 - dense_141_mae: 0.1926 - dense_141_r2_metric: -2651.3574 - dense_141_rmse: 0.2369 - dense_144_loss: 0.0811 - dense_144_mae: 0.1804 - dense_144_r2_metric: -3210.6340 - dense_144_rmse: 0.2284 - dense_147_loss: 0.0787 - dense_147_mae: 0.1747 - dense_147_r2_metric: -2986.2012 - dense_147_rmse: 0.2286 - loss: 0.4847 - val_dense_132_loss: 0.0049 - val_dense_132_mae: 0.0332 - val_dense_132_r2_metric: -21.3653 - val_dense_132_rmse: 0.0549 - val_dense_135_loss: 0.0047 - val_dense_135_mae: 0.0439 - val_dense_135_r2_metric: -95.7232 - val_dense_135_rmse: 0.0580 - val_dense_138_loss: 0.0070 - val_dens

In [47]:
csv_saving("A" , "Yes" , "Full" , "5x5" , history)

In [48]:
history = fit_models("aug_5x5.npy" , "aug/outputCE.npy" , "PH" , "multitask" , EPOCHS , BATCH_SIZE , TEST_SIZE)

Epoch 1/70
303/303 ━━━━━━━━━━━━━━━━━━━━ 21s 24ms/step - dense_150_loss: 0.0359 - dense_150_mae: 0.1377 - dense_150_r2_metric: -570.2112 - dense_150_rmse: 0.1709 - dense_152_loss: 0.0419 - dense_152_mae: 0.1418 - dense_152_r2_metric: -646.9819 - dense_152_rmse: 0.1836 - dense_154_loss: 0.1380 - dense_154_mae: 0.2565 - dense_154_r2_metric: -96.8648 - dense_154_rmse: 0.3323 - dense_156_loss: 0.0590 - dense_156_mae: 0.1844 - dense_156_r2_metric: -23.0113 - dense_156_rmse: 0.2554 - dense_158_loss: 0.0804 - dense_158_mae: 0.2058 - dense_158_r2_metric: -43.2131 - dense_158_rmse: 0.2920 - dense_160_loss: 0.0737 - dense_160_mae: 0.1956 - dense_160_r2_metric: -33.5191 - dense_160_rmse: 0.2663 - loss: 0.4290 - val_dense_150_loss: 0.0055 - val_dense_150_mae: 0.0347 - val_dense_150_r2_metric: -14.7683 - val_dense_150_rmse: 0.0522 - val_dense_152_loss: 0.0060 - val_dense_152_mae: 0.0372 - val_dense_152_r2_metric: -17.8210 - val_dense_152_rmse: 0.0566 - val_dense_154_loss: 0.0312 - val_dense_154_mae:

In [49]:
csv_saving("CE" , "Yes" , "Full" , "5x5" , history)

In [50]:
history = fit_models("aug_5x5.npy" , "aug/outputCEC.npy" , "PH" , "multitask" , EPOCHS , BATCH_SIZE , TEST_SIZE)

Epoch 1/70
303/303 ━━━━━━━━━━━━━━━━━━━━ 16s 16ms/step - dense_163_loss: 0.1826 - dense_163_mae: 0.3015 - dense_163_r2_metric: -38.9082 - dense_163_rmse: 0.4233 - dense_165_loss: 0.1080 - dense_165_mae: 0.2423 - dense_165_r2_metric: -22.4513 - dense_165_rmse: 0.3557 - dense_167_loss: 0.1064 - dense_167_mae: 0.2369 - dense_167_r2_metric: -24.1378 - dense_167_rmse: 0.3313 - dense_169_loss: 0.1507 - dense_169_mae: 0.2701 - dense_169_r2_metric: -40.2019 - dense_169_rmse: 0.3651 - dense_171_loss: 0.0844 - dense_171_mae: 0.2176 - dense_171_r2_metric: -25.9640 - dense_171_rmse: 0.3155 - dense_173_loss: 0.1080 - dense_173_mae: 0.2402 - dense_173_r2_metric: -31.8775 - dense_173_rmse: 0.3360 - loss: 0.7400 - val_dense_163_loss: 0.0491 - val_dense_163_mae: 0.1693 - val_dense_163_r2_metric: -13.2015 - val_dense_163_rmse: 0.2954 - val_dense_165_loss: 0.0440 - val_dense_165_mae: 0.1586 - val_dense_165_r2_metric: -12.6991 - val_dense_165_rmse: 0.2912 - val_dense_167_loss: 0.0409 - val_dense_167_mae: 0

In [51]:
csv_saving("CEC" , "Yes" , "Full" , "5x5" , history)

In [52]:
history = fit_models("aug_5x5.npy" , "aug/outputL%.npy" , "CE" , "multitask" , EPOCHS , BATCH_SIZE , TEST_SIZE)

Epoch 1/70
303/303 ━━━━━━━━━━━━━━━━━━━━ 15s 15ms/step - dense_176_loss: 0.1036 - dense_176_mae: 0.2562 - dense_176_r2_metric: -45.0081 - dense_176_rmse: 0.3294 - dense_178_loss: 0.2965 - dense_178_mae: 0.4452 - dense_178_r2_metric: -122.2228 - dense_178_rmse: 0.5216 - dense_180_loss: 0.6657 - dense_180_mae: 0.7228 - dense_180_r2_metric: -241.9498 - dense_180_rmse: 0.7824 - dense_182_loss: 0.4814 - dense_182_mae: 0.5735 - dense_182_r2_metric: -221.9985 - dense_182_rmse: 0.6650 - dense_184_loss: 0.3372 - dense_184_mae: 0.4860 - dense_184_r2_metric: -111.9709 - dense_184_rmse: 0.5693 - dense_186_loss: 0.1606 - dense_186_mae: 0.3187 - dense_186_r2_metric: -52.5419 - dense_186_rmse: 0.4055 - loss: 2.0451 - val_dense_176_loss: 0.0537 - val_dense_176_mae: 0.1978 - val_dense_176_r2_metric: -28.6149 - val_dense_176_rmse: 0.2417 - val_dense_178_loss: 0.0394 - val_dense_178_mae: 0.1562 - val_dense_178_r2_metric: -23.5879 - val_dense_178_rmse: 0.2221 - val_dense_180_loss: 0.0798 - val_dense_180_ma

In [53]:
csv_saving("CEC" , "Yes" , "Full" , "5x5" , history)

In [54]:
history = fit_models("aug_5x5.npy" , "aug/outputS%.npy" , "A%" , "multitask" , EPOCHS , BATCH_SIZE , TEST_SIZE)

Epoch 1/70
303/303 ━━━━━━━━━━━━━━━━━━━━ 18s 24ms/step - dense_190_loss: 0.1508 - dense_190_mae: 0.2872 - dense_190_r2_metric: -25.4555 - dense_190_rmse: 0.4158 - dense_193_loss: 0.1560 - dense_193_mae: 0.2930 - dense_193_r2_metric: -25.5952 - dense_193_rmse: 0.4141 - dense_196_loss: 0.1416 - dense_196_mae: 0.2777 - dense_196_r2_metric: -24.2836 - dense_196_rmse: 0.4117 - dense_199_loss: 0.1499 - dense_199_mae: 0.2885 - dense_199_r2_metric: -28.9458 - dense_199_rmse: 0.4164 - dense_202_loss: 0.1426 - dense_202_mae: 0.2891 - dense_202_r2_metric: -29.2346 - dense_202_rmse: 0.4019 - dense_205_loss: 0.1478 - dense_205_mae: 0.2824 - dense_205_r2_metric: -30.2646 - dense_205_rmse: 0.4121 - loss: 0.8888 - val_dense_190_loss: 0.0679 - val_dense_190_mae: 0.2164 - val_dense_190_r2_metric: -8.4064 - val_dense_190_rmse: 0.2699 - val_dense_193_loss: 0.0689 - val_dense_193_mae: 0.2165 - val_dense_193_r2_metric: -8.3352 - val_dense_193_rmse: 0.2700 - val_dense_196_loss: 0.0697 - val_dense_196_mae: 0.2

In [55]:
csv_saving("CEC" , "Yes" , "Full" , "5x5" , history)